<a href="https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
import numpy as np, pandas as pd, json, hashlib
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Colab-friendly: everything is written under the current folder
OUT, FIGS = Path("work/outputs"), Path("docs/figs")
OUT.mkdir(parents=True, exist_ok=True); FIGS.mkdir(parents=True, exist_ok=True)

# Frozen baseline thresholds (never tuned after model work starts)
STALE_DAYS, MIN_IMPR, SLIP = 180, 500, 2.0
K, SEED, DROP = 50, 7, 0.20      # label: clicks fell >= 20% in the NEXT window
USE_REAL = False                  # set True after filling in load_real()
SALT = "change-me-before-publishing"   # hashes page ids for the public paper
print(f"Settings ready | K = {K} | label = click drop of {int(DROP*100)}% or more")
"Abstract:"
"Content teams have limited time to review pages, so the order of the review queue matters. This paper asks whether a simple model can rank declining pages better than a frozen, readable rule. I compare a hand-written rule (page is old, visible, and slipping in position) with a logistic regression and a gradient-boosting challenger. I test them on held-out pages from a later time period, so the split respects time and no page appears on both sides. At precision@50, the rule scored **[0.74]**, the logistic regression [0.96], and gradient boosting [0.94], against a base rate of [0.40]. These results are observed and directional, and they support decisions. They do not show causes, and they say nothing about how Google ranks pages. The output is a ranked review queue with reason codes and a recommended action for each page."
"Question"

"Research question. Which pages should a content team review first, and can a simple model beat a frozen, readable rule at putting truly declining pages at the top of the queue?"

"The decision this supports. Reviewing a page costs time. A ranked list with reasons is more useful than an unordered one, because the team can start at the top and stop when the hours run out. If the model does not clearly beat the rule, the honest recommendation is to keep the rule, because a rule anyone can read is easier to trust and maintain."

"The rule. A page is worth reviewing if it gets traffic, is old, and is slipping in search position. Its thresholds are 500 or more impressions, 180 or more days since the last update, and a position that got worse by 2 or more places. These limits are frozen. I do not change them after model work starts, so I cannot make the model look better by weakening the rule."

Settings ready | K = 50 | label = click drop of 20% or more


'The rule. A page is worth reviewing if it gets traffic, is old, and is slipping in search position. Its thresholds are 500 or more impressions, 180 or more days since the last update, and a position that got worse by 2 or more places. These limits are frozen. I do not change them after model work starts, so I cannot make the model look better by weakening the rule.'

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
"Data:"

"Release. [EDIT: name and version of the FlyRank ML Internship release.]"

"Tables. [EDIT: the tables I aggregated with DuckDB.]"

"Date windows. Each row is one page at one snapshot in time. Features come from the window before the snapshot. The label comes from the window after it. [EDIT: exact dates of the feature and label windows.] Keeping before and after apart is what stops the model from seeing the future."

"What I excluded and why. [EDIT: for example, pages with too little traffic to measure a change. State the rule and how many pages it removed.]"

"Public-safe handling. Page identifiers are hashed with a private salt. This paper contains no client names, domains, URLs, queries, credentials, or raw exports."

"Label. A page is positive if its clicks in the next window fell by at least 20% compared with the feature window. This is an observed drop. It does not say why the drop happened."
def load_demo(n=3000, seed=SEED):
    rng = np.random.default_rng(seed); ids = [f"pg{i:05d}" for i in range(n)]
    b_impr, b_pos, b_age = rng.lognormal(6, 1.3, n), rng.uniform(1, 30, n), rng.integers(0, 900, n)
    rows = []
    for snap in (1, 2):
        age = b_age + (snap - 1) * 90
        impr = (b_impr * rng.lognormal(0, .25, n)).astype(int) + 1
        pp = np.clip(b_pos + rng.normal(0, 1, n), 1, 60)
        pn = np.clip(pp + rng.normal(.4, 2.5, n), 1, 60)
        ctr = np.clip(.3 / pn**.9 * rng.lognormal(0, .3, n), .001, .6)
        clk = (impr * ctr).astype(int)
        clk_prev = (clk * rng.lognormal(.05, .35, n)).astype(int)
        tr_ = clk / np.maximum(clk_prev, 1) - 1
        z = (-2.9 + .0035*age + .30*(pn-pp) - 1.2*np.clip(tr_, -1, 1)
             + .00015*np.minimum(impr, 5000) + .8*((age > 400) & (pn-pp > 2)))
        y_ = (rng.random(n) < 1/(1+np.exp(-z))).astype(int)
        rows.append(pd.DataFrame(dict(page_id=ids, snapshot=snap, days_since_update=age,
            impressions=impr, clicks=clk, clicks_prev=clk_prev,
            position_now=pn, position_prev=pp, label=y_)))
    return pd.concat(rows, ignore_index=True)

def load_real():
    """TODO: aggregate the warehouse release with DuckDB (starter notebook 03).
    Return the same columns as load_demo(). Features: window BEFORE each snapshot.
    Label: window AFTER. In Colab, first run:  !pip -q install duckdb
        import duckdb; con = duckdb.connect(); con.sql("INSTALL httpfs; LOAD httpfs;")
        # set your Hugging Face read token, then query 'hf://datasets/<ORG>/<DATASET>/<table>.parquet'"""
    raise NotImplementedError("Fill in load_real() with your own DuckDB aggregation.")

raw = load_real() if USE_REAL else load_demo()
print("Mode:", "REAL" if USE_REAL else "DEMO (synthetic)", "| rows:", len(raw))
raw.describe().T[["count", "mean", "min", "max"]].round(2)

Mode: DEMO (synthetic) | rows: 6000


,count,mean,min,max
snapshot,6000.0,1.50,1.0,2.00
days_since_update,6000.0,497.39,0.0,989.00
impressions,6000.0,923.47,3.0,40542.00
clicks,6000.0,42.98,0.0,3331.00
clicks_prev,6000.0,46.95,0.0,5068.00
position_now,6000.0,15.69,1.0,37.83
position_prev,6000.0,15.26,1.0,31.86
label,6000.0,0.36,0.0,1.00


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
"Methodology"

"Assumption. A click decline in the next window is a useful stand-in for this page was worth reviewing.It is an outcome I observed, not proof of a cause."

"Features. I use six features, all measured before the snapshot:"

"Days since the page was last updated"
"Log of impressions"
"Click-through rate"
"Current search position"
"Change in position"
"Click trend (recent clicks against earlier clicks)"

"Baseline. The baseline score is the page's impressions if it meets all three rule conditions (stale, visible, slipping), and zero otherwise. It has no fitted weights, so it can be read and explained in three sentences. Each flagged page carries reason codes: stale_but_visible and position_slipping."

"Models. The primary model is a standardized logistic regression, chosen because its coefficients can be read. The challenger is gradient boosting. I named the primary model before looking at test results. Picking the winner after seeing the test set would be cheating, so I report both and judge on the primary one."

"Validation design. I train on the earlier snapshot using a random 70% of pages. I test on the later snapshot using the other 30%. This split respects time because the test period comes after training. It is also grouped by page, because no page appears in both sets, so the model cannot score well by memorizing pages."

"Evaluation metric. I use precision@K with K = 50: of the 50 pages ranked highest, how many really declined later? I always report it next to the base rate and the average of random picks, because a score means little until you know what chance would give. I also report a bootstrap 95% interval for the difference between the model and the rule. It re-samples the test pages 500 times to show how much the gap could move by luck."

"Leakage checks. Five checks ran automatically and would have stopped the analysis if any failed:"

"No label, flag, or future columns are used as features."
"The test snapshot is later than the training snapshot."
"No page appears in both training and test."
"No single feature is a near-copy of the label (correlation below 0.9)."
"The rule's thresholds are the frozen constants."

"All five passed. This shows that the leaks I thought to test are absent, not that no leak exists."
d = raw.copy()
d["position_change"] = d.position_now - d.position_prev
d["click_trend"] = (d.clicks / np.maximum(d.clicks_prev, 1) - 1).clip(-1, 3)
d["log_impr"] = np.log1p(d.impressions)
d["ctr"] = d.clicks / np.maximum(d.impressions, 1)
FEATURES = ["days_since_update", "log_impr", "ctr", "position_now", "position_change", "click_trend"]

rng = np.random.default_rng(SEED)
pages = np.array(d.page_id.unique()); rng.shuffle(pages)
train_pages = set(pages[: int(.7 * len(pages))])
tr = d[(d.snapshot == 1) & d.page_id.isin(train_pages)].reset_index(drop=True)
te = d[(d.snapshot == 2) & ~d.page_id.isin(train_pages)].reset_index(drop=True)
y = te.label.to_numpy()
print(f"train {len(tr)} rows | test {len(te)} rows | test base rate {y.mean():.3f}")

checks = []
def check(name, ok):
    checks.append({"check": name, "passed": bool(ok)}); assert ok, f"LEAK: {name}"
check("no label/flag/future columns among features",
      not any(w in f.lower() for f in FEATURES for w in ["label", "flag", "future", "next_", "target"]))
check("test snapshot is later than train snapshot", te.snapshot.min() > tr.snapshot.max())
check("no page appears in both train and test", set(tr.page_id).isdisjoint(set(te.page_id)))
check("no single feature is a near-copy of the label",
      max(abs(np.corrcoef(te[f], y)[0, 1]) for f in FEATURES) < .9)
check("baseline thresholds are the frozen constants", (STALE_DAYS, MIN_IMPR, SLIP) == (180, 500, 2.0))
pd.DataFrame(checks)

train 2100 rows | test 900 rows | test base rate 0.396


,check,passed
0,no label/flag/future columns among features,True
1,test snapshot is later than train snapshot,True
2,no page appears in both train and test,True
3,no single feature is a near-copy of the label,True
4,baseline thresholds are the frozen constants,True


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
"Results (vs baseline)"

"The rule and both models were scored on the same held-out pages with the same labels."

"Method	Precision@50	Lift vs. base rate	95% interval vs. rule"
"Base rate	[0.40]	1.00	–"
"Random picks	[0.40]	1.01	–"
"Rule (frozen baseline)	[0.74]	1.87	–"
"Logistic regression (primary)	[0.96]	2.43	[0.10, 0.34]"
"Gradient boosting (challenger)	[0.94]	2.37	[0.07, 0.33]"

"Figure 1 shows these values as bars. Figure 2 shows precision across K from 10 to 200, with the base rate as a dashed line."

"Reading the table. On this split, the primary model's precision@50 was higher than the rule's, and the bootstrap interval for the difference excludes zero [replace with what your own interval shows]. Both models beat random picking by a wide margin. The two models are close to each other, so I do not claim one is better than the other."

"Things to keep in view. The rule flagged only [74] of the [900] test pages, so at larger K the rule starts ranking pages with a score of zero. The interval is [0.24] wide, which means the size of the gap is uncertain even if its direction is not. The largest logistic coefficients, in order, were [list your top features and their signs]. These show association within this data, not cause."
def precision_at_k(s, y_, k):
    return np.asarray(y_)[np.argsort(-np.asarray(s), kind="stable")[:k]].mean()

def rule_score(x):
    return ((x.days_since_update >= STALE_DAYS).astype(int) * (x.impressions >= MIN_IMPR).astype(int)
            * (x.position_change >= SLIP).astype(int) * x.impressions).to_numpy()

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(tr[FEATURES], tr.label)
hgb = HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200,
                                     random_state=SEED).fit(tr[FEATURES], tr.label)
S = {"rule": rule_score(te), "lr": lr.predict_proba(te[FEATURES])[:, 1],
     "hgb": hgb.predict_proba(te[FEATURES])[:, 1]}

base = float(y.mean())
rr = np.random.default_rng(0)
p_random = float(np.mean([y[rr.permutation(len(y))[:K]].mean() for _ in range(500)]))
P = {m: float(precision_at_k(s, y, K)) for m, s in S.items()}
KS = [10, 25, 50, 100, 200]
curve = {m: [float(precision_at_k(s, y, k)) for k in KS] for m, s in S.items()}

def boot_diff(a, b, k, B=500):
    r = np.random.default_rng(1); n = len(y); out = []
    for _ in range(B):
        i = r.integers(0, n, n)
        out.append(precision_at_k(a[i], y[i], k) - precision_at_k(b[i], y[i], k))
    return [float(v) for v in np.percentile(out, [2.5, 97.5])]

ci_lr, ci_hgb = boot_diff(S["lr"], S["rule"], K), boot_diff(S["hgb"], S["rule"], K)
n_flag = int((S["rule"] > 0).sum())

table = pd.DataFrame({
    "method": ["Base rate", "Random picks", "Rule (frozen baseline)", "Logistic regression (primary)", "Gradient boosting (challenger)"],
    f"precision@{K}": [base, p_random, P["rule"], P["lr"], P["hgb"]],
    "lift vs base rate": [1.0, p_random/base, P["rule"]/base, P["lr"]/base, P["hgb"]/base],
    "95% interval vs rule": ["", "", "", f"[{ci_lr[0]:.2f}, {ci_lr[1]:.2f}]", f"[{ci_hgb[0]:.2f}, {ci_hgb[1]:.2f}]"],
}).round(3)
display(table)

if ci_lr[0] > 0:
    verdict = "was higher than the rule's, and the interval for the difference excludes zero"
elif P["lr"] > P["rule"]:
    verdict = "was directionally higher than the rule's, but the interval includes zero"
else:
    verdict = "did not beat the rule"
display(Markdown(f"**Reading the table.** On this split, the primary model's precision@{K} ({P['lr']:.2f}) {verdict} "
                 f"(rule: {P['rule']:.2f}; base rate: {base:.2f}). "
                 + ("*These numbers come from synthetic demo data.*" if not USE_REAL else "")))

,method,precision@50,lift vs base rate,95% interval vs rule
0,Base rate,0.396,1.000,
1,Random picks,0.401,1.013,
2,Rule (frozen baseline),0.740,1.871,
3,Logistic regression (primary),0.960,2.427,"[0.10, 0.34]"
4,Gradient boosting (challenger),0.940,2.376,"[0.07, 0.33]"


**Reading the table.** On this split, the primary model's precision@50 (0.96) was higher than the rule's, and the interval for the difference excludes zero (rule: 0.74; base rate: 0.40). *These numbers come from synthetic demo data.*

## 5. Limitations

*What this work cannot claim.*

In [ ]:
"Limitations"

"This work cannot claim the following:"

"It is observational. The results are observed associations on one split. Precision@K is directional, decision-support evidence."
"The label is a proxy. A click decline can come from seasonality, shifts in demand, or tracking changes, not only content quality."
"No claim about Google. This paper does not show how Google's algorithm works."
"No causal claim. It does not show that refreshing a page causes recovery. A refresh was not randomized or tested here."
"Leakage checks are limited. They cover the leaks I thought to test, not every possible leak."
"One split, one estimate. Repeating the analysis over more time windows would show how stable the result is."
"Weak picks exist. In the top 20, [0] pages did not show a later decline [replace; with real data expect some]. A flagged page may be evergreen, seasonal, or affected by a one-time spike, so each pick needs a human check."
print(f"Test pages: {len(te)} | pages with a later decline: {int(y.sum())}")
print(f"The rule flagged only {n_flag} test rows; beyond that it ranks zero-score pages.")
print(f"Interval width for (logistic regression - rule) at K={K}: {ci_lr[1]-ci_lr[0]:.2f} (wide = unstable)")
if P["lr"] <= P["rule"]:
    print("The model did NOT beat the rule on this split: recommend keeping the rule.")
elif ci_lr[0] <= 0:
    print("The model is directionally ahead, but the interval includes zero.")
else:
    print("The model is ahead and the interval excludes zero on this split (still directional).")

Test pages: 900 | pages with a later decline: 356
The rule flagged only 74 test rows; beyond that it ranks zero-score pages.
Interval width for (logistic regression - rule) at K=50: 0.24 (wide = unstable)
The model is ahead and the interval excludes zero on this split (still directional).


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
"Ranked recommendations"

"Test pages are ranked by the primary model. Each page receives reason codes and an action, as shown in the playbook below. Page identifiers are hashed."

"Reason code	What it means	Recommended action"
"stale_but_visible	Old page that still gets impressions	Refresh the content and re-check that it matches search intent"
"position_slipping	Position got worse by 2 or more places	Review competing results and on-page relevance"
"clicks_declining	Recent clicks are at least 20% below earlier clicks	Review the title and description, then monitor"
"none	No condition met	Monitor"

"How to use the queue. Start at rank 1 and work down as far as review time allows. Treat each recommendation as a prompt for human review, not an instruction to change the page. Check the top 20 by hand first. If the model is not clearly ahead of the rule on your real data, use the rule's queue instead."

"Table 2 in the deployed paper lists the top 10 pages (hashed reference, reason codes, action)."
ACTIONS = {"stale_but_visible": "Refresh content; re-check search-intent match",
           "position_slipping": "Review competing results and on-page relevance",
           "clicks_declining": "Review title and description; monitor",
           "none": "Monitor"}
def reasons(r):
    c = []
    if r.days_since_update >= STALE_DAYS and r.impressions >= MIN_IMPR: c.append("stale_but_visible")
    if r.position_change >= SLIP: c.append("position_slipping")
    if r.click_trend <= -DROP: c.append("clicks_declining")
    return c or ["none"]

rec = te.assign(score=S["lr"]).sort_values("score", ascending=False).reset_index(drop=True)
rec["rank"] = rec.index + 1
rec["reason_codes"] = rec.apply(lambda r: "|".join(reasons(r)), axis=1)
rec["action"] = rec.reason_codes.map(lambda s: ACTIONS[s.split("|")[0]])
rec["page_ref"] = rec.page_id.map(lambda p: hashlib.sha1((SALT + p).encode()).hexdigest()[:8])
cols = ["rank", "page_ref", "score", "reason_codes", "action", "label"]
rec[cols].to_csv(OUT / "ranked_recommendations.csv", index=False)
top = rec.head(20)
print(f"Top 20: {int((top.label == 0).sum())} weak picks (no later decline). Review each by hand.")
rec[cols].head(20)

Top 20: 0 weak picks (no later decline). Review each by hand.


,rank,page_ref,score,reason_codes,action,label
0,1,9f2447cd,0.980273,position_slipping|clicks_declining,Review competing results and on-page relevance,1
1,2,b4d77252,0.970152,stale_but_visible|position_slipping|clicks_dec...,Refresh content; re-check search-intent match,1
2,3,83d12bf2,0.968335,position_slipping|clicks_declining,Review competing results and on-page relevance,1
3,4,921b109d,0.961372,position_slipping|clicks_declining,Review competing results and on-page relevance,1
4,5,c024d41b,0.961327,stale_but_visible|position_slipping,Refresh content; re-check search-intent match,1
5,6,26cc4906,0.957793,position_slipping,Review competing results and on-page relevance,1
6,7,7c1f5334,0.951560,stale_but_visible|position_slipping,Refresh content; re-check search-intent match,1
7,8,6f017194,0.949131,position_slipping|clicks_declining,Review competing results and on-page relevance,1
8,9,d2b6c95e,0.948866,position_slipping|clicks_declining,Review competing results and on-page relevance,1
9,10,0de7f8e7,0.948786,position_slipping,Review competing results and on-page relevance,1


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
"Artifacts the paper embeds"

"The deployed page shows these items, all generated from the analysis:"

"Figure 1. Bar chart of precision@50 for the base rate, random picks, the rule, logistic regression, and gradient boosting."
"Figure 2. Line chart of precision across K (10, 25, 50, 100, 200), with the base rate as a dashed line."
"Table 1. The results table in Section 4."
"Table 2. The top 10 ranked pages with hashed references, reason codes, and actions."
"Table 3. The leakage-check results."
"Table 4. The standardized logistic coefficients, largest first."
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots(figsize=(6.4, 3.4))
names = ["Base rate", "Random", "Rule (frozen)", "Logistic reg.", "Grad. boosting"]
vals = [base, p_random, P["rule"], P["lr"], P["hgb"]]
ax.bar(names, vals, color=["#bbb", "#bbb", "#7a8aa0", "#2563eb", "#60a5fa"])
for i, v in enumerate(vals): ax.text(i, v + .01, f"{v:.2f}", ha="center", fontsize=10)
ax.set_ylabel(f"precision@{K}"); ax.set_ylim(0, 1); plt.xticks(rotation=15)
plt.title("Figure 1. Precision@K on the held-out split"); plt.tight_layout()
plt.savefig(FIGS / "precision_at_k.png", dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(6.4, 3.4))
for m, lab, c in [("rule", "Rule (frozen)", "#7a8aa0"), ("lr", "Logistic reg.", "#2563eb"), ("hgb", "Grad. boosting", "#60a5fa")]:
    ax.plot(KS, curve[m], marker="o", label=lab, color=c)
ax.axhline(base, ls="--", color="#999", label="Base rate")
ax.set_xlabel("K (pages reviewed)"); ax.set_ylabel("precision@K"); ax.set_ylim(0, 1); ax.legend(frameon=False)
plt.title("Figure 2. Precision across K"); plt.tight_layout()
plt.savefig(FIGS / "precision_curve.png", dpi=160); plt.show()

coefs = sorted(zip(FEATURES, lr.named_steps["logisticregression"].coef_[0]), key=lambda t: -abs(t[1]))
display(pd.DataFrame(coefs, columns=["feature", "standardized coefficient"]).round(3))

results = {
    "demo": not USE_REAL, "k": K, "n_train": len(tr), "n_test": len(te),
    "base_rate": base, "p_random": p_random, "p": P, "ci_lr_vs_rule": ci_lr,
    "ci_hgb_vs_rule": ci_hgb, "n_rule_flagged": n_flag, "ks": KS, "curve": curve,
    "leakage": checks, "coefs": [[f, float(c)] for f, c in coefs],
    "top20_decline_rate": float(top.label.mean()), "top20_weak": int((top.label == 0).sum()),
    "top10": [{"rank": int(r["rank"]), "page_ref": r.page_ref, "reasons": r.reason_codes, "action": r.action}
              for _, r in rec.head(10).iterrows()],
}
(OUT / "results.json").write_text(json.dumps(results, indent=2))
print("Saved: results.json, ranked_recommendations.csv, precision_at_k.png, precision_curve.png")

# Colab only: zip and download everything the paper page needs
try:
    from google.colab import files
    import shutil
    shutil.make_archive("paper_artifacts", "zip", ".", "work/outputs")
    shutil.make_archive("paper_figs", "zip", ".", "docs/figs")
    files.download("paper_artifacts.zip"); files.download("paper_figs.zip")
except ImportError:
    print("(Not running in Colab: files are in work/outputs and docs/figs.)")

,feature,standardized coefficient
0,days_since_update,1.104
1,position_change,0.990
2,click_trend,-0.532
3,position_now,-0.080
4,log_impr,0.073
5,ctr,-0.009


Saved: results.json, ranked_recommendations.csv, precision_at_k.png, precision_curve.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.